In [3]:
"""
Cross-Project Vulnerability Detection on Reveal
------------------------------------------------
Train on Debian -> Test on Chrome
Domain Adaptation (DANN) + XGBoost as final classifier
No oversampling
"""

import json
import math
import random
import traceback
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from torch.autograd import Function
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from xgboost import XGBClassifier

# =========================================================
# Config
# =========================================================
SEED            = 42
TRAIN_PROJECT   = "debian"
TEST_PROJECT    = "chrome"
DEBIAN_EMB_FILE = "../../../../embedding/reveal/codebert/debian_embeddings.npy"
DEBIAN_LBL_FILE = "../../../../embedding/reveal/codebert/debian_labels.npy"
CHROME_EMB_FILE = "../../../../embedding/reveal/codebert/chrome_embeddings.npy"
CHROME_LBL_FILE = "../../../../embedding/reveal/codebert/chrome_labels.npy"
OUTPUT_DIR      = "results/no_oversampling"

DANN_EPOCHS            = 50
DANN_BATCH_SIZE        = 64
LEARNING_RATE          = 1e-5
DOMAIN_LOSS_WEIGHT_MAX = 1.0
VALIDATION_SIZE        = 0.2
NUM_RUNS               = 3
THRESHOLD              = 0.5
METHOD_VERSION         = "dann_xgboost_v1"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
else:
    DEVICE = "cpu"


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)


def atomic_json_dump(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, allow_nan=False)
        f.flush()
    tmp.replace(path)


# =========================================================
# Gradient Reversal Layer
# =========================================================
class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


class GradientReversalLayer(nn.Module):
    def forward(self, x, alpha):
        return GradientReversalFunction.apply(x, alpha)


# =========================================================
# Architecture: 64-dim bottleneck FeatureExtractor + linear ClassifierHead
# + DomainDiscriminator
# =========================================================
class FeatureExtractor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

    def forward(self, x):
        return self.network(x)  # 64-dim domain-invariant features


class ClassifierHead(nn.Module):
    def __init__(self, feature_dim=64):
        super().__init__()
        self.network = nn.Linear(feature_dim, 1)

    def forward(self, features):
        return self.network(features).squeeze(-1)


class DomainDiscriminator(nn.Module):
    def __init__(self, feature_dim=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, features):
        return self.network(features).squeeze(-1)


# =========================================================
# DANN training
# =========================================================
def train_dann(x_source, y_source, x_target, seed, pos_weight_scalar):
    set_seed(seed)
    stratify = y_source if len(np.unique(y_source)) > 1 else None
    x_train, x_val, y_train, y_val = train_test_split(
        x_source, y_source,
        test_size=VALIDATION_SIZE,
        random_state=seed,
        stratify=stratify,
    )

    feature_extractor = FeatureExtractor(x_source.shape[1]).to(DEVICE)
    classifier         = ClassifierHead(feature_dim=64).to(DEVICE)
    discriminator      = DomainDiscriminator(feature_dim=64).to(DEVICE)
    grl = GradientReversalLayer()

    params = (list(feature_extractor.parameters()) +
              list(classifier.parameters()) +
              list(discriminator.parameters()))
    optimizer = torch.optim.Adam(params, lr=LEARNING_RATE)

    pos_weight_t = torch.tensor(pos_weight_scalar, dtype=torch.float32, device=DEVICE)
    classification_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)
    domain_loss_fn = nn.BCEWithLogitsLoss()

    source_dataset = TensorDataset(
        torch.tensor(x_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32)
    )
    target_dataset = TensorDataset(torch.tensor(x_target, dtype=torch.float32))
    source_loader = DataLoader(source_dataset, batch_size=DANN_BATCH_SIZE, shuffle=True)
    target_loader = DataLoader(target_dataset, batch_size=DANN_BATCH_SIZE, shuffle=True)

    x_val_t = torch.tensor(x_val, dtype=torch.float32, device=DEVICE)
    y_val_t = torch.tensor(y_val, dtype=torch.float32, device=DEVICE)

    total_steps = max(1, DANN_EPOCHS * len(source_loader) - 1)
    global_step = 0
    history = []

    for epoch in tqdm(range(DANN_EPOCHS), desc=f"DANN seed={seed}", leave=False):
        feature_extractor.train()
        classifier.train()
        discriminator.train()

        target_iter = iter(target_loader)
        classification_losses, domain_losses = [], []

        for xs_batch, ys_batch in source_loader:
            try:
                (xt_batch,) = next(target_iter)
            except StopIteration:
                target_iter = iter(target_loader)
                (xt_batch,) = next(target_iter)

            xs_batch = xs_batch.to(DEVICE)
            ys_batch = ys_batch.to(DEVICE)
            xt_batch = xt_batch.to(DEVICE)

            progress = global_step / total_steps
            alpha = DOMAIN_LOSS_WEIGHT_MAX * (2.0 / (1.0 + math.exp(-10.0 * progress)) - 1.0)

            optimizer.zero_grad(set_to_none=True)

            source_features = feature_extractor(xs_batch)
            target_features = feature_extractor(xt_batch)

            vulnerability_logits = classifier(source_features)
            classification_loss = classification_loss_fn(vulnerability_logits, ys_batch)

            source_domain_logits = discriminator(grl(source_features, alpha))
            target_domain_logits = discriminator(grl(target_features, alpha))
            source_domain_labels = torch.zeros(source_domain_logits.shape[0], dtype=torch.float32, device=DEVICE)
            target_domain_labels = torch.ones(target_domain_logits.shape[0], dtype=torch.float32, device=DEVICE)

            source_domain_loss = domain_loss_fn(source_domain_logits, source_domain_labels)
            target_domain_loss = domain_loss_fn(target_domain_logits, target_domain_labels)
            domain_loss = 0.5 * (source_domain_loss + target_domain_loss)

            loss = classification_loss + domain_loss
            loss.backward()
            optimizer.step()

            classification_losses.append(classification_loss.item())
            domain_losses.append(domain_loss.item())
            global_step += 1

        feature_extractor.eval()
        classifier.eval()
        discriminator.eval()
        with torch.no_grad():
            val_logits = classifier(feature_extractor(x_val_t))
            val_loss = classification_loss_fn(val_logits, y_val_t).item()
            val_prob = torch.sigmoid(val_logits)
            val_pred = (val_prob >= THRESHOLD).long().cpu().numpy()
            val_f1 = f1_score(y_val, val_pred, zero_division=0)

        epoch_result = {
            "epoch": epoch + 1,
            "classification_loss": float(np.mean(classification_losses)),
            "domain_loss": float(np.mean(domain_losses)),
            "source_validation_loss": float(val_loss),
            "source_validation_f1": float(val_f1),
            "grl_alpha": float(alpha),
        }
        history.append(epoch_result)
        print(f"      epoch={epoch + 1}/{DANN_EPOCHS} "
              f"cls={epoch_result['classification_loss']:.4f} "
              f"domain={epoch_result['domain_loss']:.4f} "
              f"val={val_loss:.4f} val_f1={val_f1:.4f} alpha={alpha:.4f}")

    return feature_extractor, classifier, discriminator, history


# =========================================================
# XGBoost trained on DANN's extracted (domain-invariant) features
# =========================================================
def extract_features(feature_extractor, X, batch_size=256):
    feature_extractor.eval()
    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    outputs = []
    with torch.no_grad():
        for (x_batch,) in loader:
            x_batch = x_batch.to(DEVICE)
            outputs.append(feature_extractor(x_batch).cpu().numpy())
    return np.concatenate(outputs) if outputs else np.empty((0, 64), dtype=np.float32)


def classifier_probabilities(feature_extractor, classifier, X, batch_size=256):
    feature_extractor.eval()
    classifier.eval()
    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    outputs = []
    with torch.no_grad():
        for (x_batch,) in loader:
            x_batch = x_batch.to(DEVICE)
            logits = classifier(feature_extractor(x_batch))
            outputs.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(outputs) if outputs else np.empty(0, dtype=np.float32)


def train_xgboost(x_train_feat, y_train, class_weight_dict, sample_weights, seed):
    scale_pos_weight = class_weight_dict[1] / class_weight_dict[0]
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=seed,
    )
    model.fit(x_train_feat, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Evaluation
# =========================================================
def evaluate(y_true, probabilities):
    predictions = (probabilities >= THRESHOLD).astype(np.int32)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    accuracy = accuracy_score(y_true, predictions)
    precision = precision_score(y_true, predictions, zero_division=0)
    recall = recall_score(y_true, predictions, zero_division=0)
    f1 = f1_score(y_true, predictions, zero_division=0)
    roc_auc = roc_auc_score(y_true, probabilities) if len(np.unique(y_true)) > 1 else None
    pr_auc = average_precision_score(y_true, probabilities) if len(np.unique(y_true)) > 1 else None
    tpr = tp / (tp + fn) if tp + fn else 0.0
    tnr = tn / (tn + fp) if tn + fp else 0.0
    g_mean = math.sqrt(tpr * tnr)
    pf = fp / (fp + tn) if fp + tn else 0.0
    metrics = {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": None if roc_auc is None else float(roc_auc),
        "pr_auc": None if pr_auc is None else float(pr_auc),
        "g_mean": float(g_mean),
        "pf": float(pf),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }
    return metrics, predictions


def save_confusion_matrix(metrics, path, title="Confusion Matrix (Chrome Test Set) - DANN + XGBoost", show=False):
    cm_values = metrics["confusion_matrix"]
    cm = np.asarray([[cm_values["tn"], cm_values["fp"]], [cm_values["fn"], cm_values["tp"]]])
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(cm, interpolation="nearest")
    ax.set_title(title)
    plt.colorbar(image, ax=ax)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])
    thresh = cm.max() / 2 if cm.max() else 0
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", color=color)
    ax.set_ylabel("Actual Label"); ax.set_xlabel("Predicted Label")
    fig.tight_layout()
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=300)
    if show:
        plt.show()
    plt.close(fig)
    print(f"Confusion matrix saved as {path}")


def cpu_state_dict(module):
    return {key: value.detach().cpu() for key, value in module.state_dict().items()}


# =========================================================
# Single run: one DANN training + one XGBoost fit
# =========================================================
def run_single(run_number, X_source, y_source, X_target, y_target, scaler, class_weight_dict, output_dir):
    seed = SEED + run_number - 1
    run_dir = Path(output_dir) / f"run_{run_number:02d}"
    run_dir.mkdir(parents=True, exist_ok=True)
    started_at = now_utc()

    pos_weight_scalar = class_weight_dict[1] / class_weight_dict[0]

    feature_extractor, classifier, discriminator, history = train_dann(
        X_source, y_source, X_target, seed, pos_weight_scalar
    )

    # Extract domain-invariant features for train + target
    train_features = extract_features(feature_extractor, X_source)
    target_features = extract_features(feature_extractor, X_target)

    # Confidence-based sample weights from the DANN's own classifier head
    train_probs = classifier_probabilities(feature_extractor, classifier, X_source)
    sample_weights = np.where(y_source == 1, train_probs, 1 - train_probs)

    model_xgb = train_xgboost(train_features, y_source, class_weight_dict, sample_weights, seed)
    probabilities = model_xgb.predict_proba(target_features)[:, 1]

    metrics, predictions = evaluate(y_target, probabilities)
    finished_at = now_utc()

    result = {
        "method_version": METHOD_VERSION,
        "oversampling_method": "No Oversampling",
        "train": TRAIN_PROJECT,
        "test": TEST_PROJECT,
        "run": run_number,
        "seed": seed,
        "final_classifier": "XGBoost on DANN-extracted features",
        "n_source_samples": int(len(y_source)),
        "n_target_samples": int(len(y_target)),
        "n_source_vulnerable": int(y_source.sum()),
        "n_target_vulnerable": int(y_target.sum()),
        "source_positive_weight": float(pos_weight_scalar),
        "threshold": THRESHOLD,
        "metrics": metrics,
        "training_history": history,
        "started_at": started_at,
        "finished_at": finished_at,
    }
    atomic_json_dump(result, run_dir / "result.json")
    np.savez_compressed(
        run_dir / "predictions.npz",
        target=y_target, probability=probabilities, prediction=predictions
    )
    torch.save(
        {
            "method_version": METHOD_VERSION,
            "train": TRAIN_PROJECT,
            "test": TEST_PROJECT,
            "run": run_number,
            "seed": seed,
            "feature_extractor": cpu_state_dict(feature_extractor),
            "classifier": cpu_state_dict(classifier),
            "domain_discriminator": cpu_state_dict(discriminator),
            "scaler_mean": scaler.mean_,
            "scaler_scale": scaler.scale_,
            "threshold": THRESHOLD,
        },
        run_dir / "model.pt",
    )
    save_confusion_matrix(metrics, run_dir / "confusion_matrix.png")
    print(f"      Saved run {run_number} to {run_dir}")

    del feature_extractor, classifier, discriminator, model_xgb
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()

    return result


# =========================================================
# Ensemble aggregation
# =========================================================
def aggregate_ensemble(run_results, output_dir, y_target):
    prob_arrays = []
    for r in run_results:
        npz_path = Path(output_dir) / f"run_{r['run']:02d}" / "predictions.npz"
        data = np.load(npz_path)
        prob_arrays.append(data["probability"])

    ensemble_probability = np.mean(np.stack(prob_arrays, axis=0), axis=0)
    ensemble_metrics, ensemble_predictions = evaluate(y_target, ensemble_probability)

    per_run_metric_names = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "g_mean", "pf"]
    per_run_summary = {}
    for metric in per_run_metric_names:
        values = [r["metrics"][metric] for r in run_results if r["metrics"][metric] is not None]
        per_run_summary[metric] = {
            "mean": float(np.mean(values)) if values else None,
            "std": float(np.std(values, ddof=1)) if len(values) > 1 else (0.0 if values else None),
        }

    return {
        "method_version": METHOD_VERSION,
        "method": "DANN + XGBoost",
        "train": TRAIN_PROJECT,
        "test": TEST_PROJECT,
        "num_completed_runs": len(run_results),
        "ensemble_metrics": ensemble_metrics,
        "per_run_metric_summary": per_run_summary,
        "runs": run_results,
        "updated_at": now_utc(),
    }, ensemble_predictions


# =========================================================
# Main
# =========================================================
def main():
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Device: {DEVICE}")
    print(f"Method: {METHOD_VERSION}")
    print(f"Train project: {TRAIN_PROJECT} -> Test project: {TEST_PROJECT}")

    print("\n[1/4] Loading embeddings...")
    X_source = np.load(DEBIAN_EMB_FILE).astype(np.float32)
    y_source = np.load(DEBIAN_LBL_FILE).astype(np.int32)
    X_target = np.load(CHROME_EMB_FILE).astype(np.float32)
    y_target = np.load(CHROME_LBL_FILE).astype(np.int32)
    print(f"      Debian (source): {X_source.shape} vulnerable={int(y_source.sum())} benign={int((y_source == 0).sum())}")
    print(f"      Chrome (target): {X_target.shape} vulnerable={int(y_target.sum())} benign={int((y_target == 0).sum())}")

    if len(np.unique(y_source)) < 2:
        raise ValueError("Source pool (debian) does not contain both classes.")

    print("\n[2/4] Normalizing embeddings...")
    scaler = StandardScaler()
    X_source = scaler.fit_transform(X_source).astype(np.float32)
    X_target = scaler.transform(X_target).astype(np.float32)

    print("\n[3/4] Computing class weights...")
    classes = np.unique(y_source)
    class_weights_arr = compute_class_weight(class_weight="balanced", classes=classes, y=y_source)
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, class_weights_arr)}
    print(f"      Class weights (source pool): {class_weight_dict}")

    print("\n[4/4] Running DANN + XGBoost ensemble runs...")
    run_results = []
    for run_number in range(1, NUM_RUNS + 1):
        print(f"\n      Run {run_number}/{NUM_RUNS}")
        try:
            result = run_single(run_number, X_source, y_source, X_target, y_target,
                                 scaler, class_weight_dict, output_dir)
            run_results.append(result)
        except Exception as exc:
            print(f"      [RUN FAILED] run {run_number}: {exc}")
            traceback.print_exc()
            continue

    if not run_results:
        raise RuntimeError(f"All {NUM_RUNS} runs failed.")

    print("\nAggregating ensemble results...")
    ensemble_summary, ensemble_predictions = aggregate_ensemble(run_results, output_dir, y_target)
    atomic_json_dump(ensemble_summary, output_dir / "dann_xgboost_chrome.json")
    save_confusion_matrix(ensemble_summary["ensemble_metrics"],
                          output_dir / "confusion_matrix_dann_xgboost.png")

    m = ensemble_summary["ensemble_metrics"]
    print("\n=== Ensemble Evaluation Results (Chrome Test Set) ===")
    print(f"Accuracy:   {m['accuracy']:.3f}")
    print(f"Precision:  {m['precision']:.3f}")
    print(f"Recall:     {m['recall']:.3f}")
    print(f"F1-score:   {m['f1']:.3f}")
    print(f"ROC-AUC:    {m['roc_auc']:.3f}" if m['roc_auc'] is not None else "ROC-AUC:    N/A")
    print(f"PR-AUC:     {m['pr_auc']:.3f}" if m['pr_auc'] is not None else "PR-AUC:     N/A")
    print(f"G-mean:     {m['g_mean']:.3f}")
    print(f"PF value:   {m['pf']:.3f}")
    print(f"\nResults saved to {output_dir / 'dann_xgboost_chrome.json'}")
    print(f"Confusion matrix saved to {output_dir / 'confusion_matrix_dann_xgboost.png'}")


if __name__ == "__main__":
    main()

Device: mps
Method: dann_xgboost_v1
Train project: debian -> Test project: chrome

[1/4] Loading embeddings...
      Debian (source): (18298, 768) vulnerable=1415 benign=16883
      Chrome (target): (4436, 768) vulnerable=825 benign=3611

[2/4] Normalizing embeddings...

[3/4] Computing class weights...
      Class weights (source pool): {0: 0.5419060593496416, 1: 6.465724381625441}

[4/4] Running DANN + XGBoost ensemble runs...

      Run 1/3


DANN seed=42:   2%|█▍                                                                    | 1/50 [00:00<00:46,  1.05it/s]

      epoch=1/50 cls=1.2690 domain=0.6945 val=1.2508 val_f1=0.1687 alpha=0.0992


DANN seed=42:   4%|██▊                                                                   | 2/50 [00:01<00:48,  1.01s/it]

      epoch=2/50 cls=1.2398 domain=0.6944 val=1.2147 val_f1=0.2601 alpha=0.1970


...

      epoch=48/50 cls=0.7382 domain=0.6939 val=0.9743 val_f1=0.3306 alpha=0.9999


DANN seed=44:  98%|███████████████████████████████████████████████████████████████████▌ | 49/50 [00:50<00:01,  1.02s/it]

      epoch=49/50 cls=0.7215 domain=0.6926 val=0.9785 val_f1=0.3275 alpha=0.9999


      epoch=50/50 cls=0.7205 domain=0.6929 val=0.9787 val_f1=0.3339 alpha=0.9999


Confusion matrix saved as results/no_oversampling/run_03/confusion_matrix.png
      Saved run 3 to results/no_oversampling/run_03

Aggregating ensemble results...
Confusion matrix saved as results/no_oversampling/confusion_matrix_dann_xgboost.png

=== Ensemble Evaluation Results (Chrome Test Set) ===
Accuracy:   0.740
Precision:  0.281
Recall:     0.257
F1-score:   0.269
ROC-AUC:    0.598
PR-AUC:     0.256
G-mean:     0.467
PF value:   0.150

Results saved to results/no_oversampling/dann_xgboost_chrome.json
Confusion matrix saved to results/no_oversampling/confusion_matrix_dann_xgboost.png
